[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/D.%20Integrated%20Tactical%20%26%20Pre-Planning%20Problems%20%28Connecting%20the%20Silos%29/27.%20The%20Integrated%20Berth%20%26%20Crane%20Allocation%20Problem%20%28BAP-QCAP%29/P27-Tier-4_executed.ipynb)

---


# 27. The Integrated Berth and Crane Allocation Problem (BAP-QCAP)

## Tier 4: Deep Reinforcement Learning with a Dueling DQN

### Goal

Build and train a Dueling Deep Q-Network (Dueling DQN) agent that selects a vessel, a berth position, and a crane count in a simulated port environment. The notebook demonstrates the complete learning workflow: defining the decision process, simulating episodes, learning from stored experience, and inspecting training and evaluation results.

### Inputs and outputs

The example uses a 1,200 m quay, 15 cranes, a 48-hour planning horizon, and randomly generated vessel characteristics. The training function returns the agent, its environment, and episode-level training metrics. Evaluation and policy-analysis functions report measurements produced by their own simulated episodes; they do not assume a particular performance level.

### Model assumptions and scope

- Vessel lengths, workloads, arrival times, hourly costs, maximum crane counts, and priorities are generated as specified in the code. Priority is recorded in the state but is not used in the reward.
- Decisions are made sequentially. A vessel's service begins no earlier than the current simulation time or its arrival time. The simulation advances to the end of that service before the next decision. Consequently, this teaching environment does not model concurrent vessel service or simultaneous crane assignments.
- A vessel is assigned one berth position and a number of cranes. The selected position is checked against quay occupancy, and service productivity accounts for crane interference.
- Arrivals are generated at the beginning of each episode. There are no vessel arrivals during an episode, breakdowns, tides, setup times, or uncertainty in service productivity.
- The discretized action space contains 15 vessel-ID slots, 30 berth-position bins, and 5 crane-count choices. Invalid actions are excluded when choosing an action and when forming the next-state value target.
- The reported evaluation describes this simulator and these generated scenarios. It is not a comparison with a rule-based baseline, historical data, or a guarantee of optimality.

### Decision process and Dueling DQN

A Markov decision process (MDP) represents a sequential problem using a state, an available action, a reward, and a transition to a new state. Here, the state summarizes the pending vessels, berth occupancy, crane availability, and simulation time. An action assigns a pending vessel to a berth position and crane count. The reward is the negative cost of waiting and service, with the stated bonuses. The transition records the assignment and advances the simulation clock.

The Q-value, \(Q(s,a)\), estimates the discounted return from taking action \(a\) in state \(s\) and then following the policy. The Dueling DQN estimates a state value \(V(s)\) and action advantages \(A(s,a)\) separately, then combines them:

$$
Q(s,a) = V(s) + \left(A(s,a) - \frac{1}{|\mathcal{A}|}\sum_{a' \in \mathcal{A}} A(s,a')\right).
$$

The mean subtraction makes the value-and-advantage decomposition identifiable: adding the same constant to every advantage does not change the relative action values. In the code, `DuelingDQN.forward` implements this combination.

Experience replay stores transitions and samples them in batches so that updates do not rely only on consecutive, highly related decisions. A separate target network supplies the next-state estimate in the Q-learning target:

$$
y = r + \gamma(1-d)\max_{a' \in \mathcal{A}(s')} Q_{\text{target}}(s',a'),
\qquad
L = \frac{1}{B}\sum_{i=1}^{B}\left(Q_{\text{online}}(s_i,a_i)-y_i\right)^2.
$$

Here, \(r\) is the observed reward, γ is the discount factor, \(d\) is 1 for a terminal transition and 0 otherwise, \(\mathcal{A}(s')\) is the set of valid next actions, and \(B\) is the replay-batch size. The target network is copied from the online network periodically. Epsilon-greedy action selection explores a random valid action with probability ε and otherwise selects the valid action with the largest estimated Q-value.

The state vector has 130 features. Its fixed size allows it to be passed to a neural network, while the encoding limits detailed vessel features to the first five pending vessels. The model has 2,250 discrete action indices; valid actions are generated from the current state rather than treated as available in every state. The network output and replay computation therefore grow with the fixed state and action dimensions. The simulator itself generates a small finite set of candidate actions by enumerating vessel, position, and crane choices.

### Learning and evaluation

Training uses the concrete instance below and 500 episodes. Each episode generates a new random set of vessels. The plots show the resulting episode rewards, exploration rate, training loss, and vessels served. These measurements are diagnostic: a reward curve alone does not establish convergence or prove that a policy is optimal. Evaluation and scenario analysis use additional generated episodes and report only the values actually measured when the notebook is run.

In [ ]:
from collections import deque, namedtuple
from dataclasses import dataclass
import random
import time
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

# Fixed seeds make the generated training sequence reproducible for a fixed runtime and library setup.
np.random.seed(42)
random.seed(42)
torch.manual_seed(42)

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

print("Libraries imported successfully.")

In [ ]:
@dataclass
class Vessel:
    """A vessel requiring berth and crane services."""

    id: int
    length: float
    workload: float
    arrival_time: float
    cost_per_hour: float = 1000.0
    max_cranes: int = 5
    priority: float = 1.0


@dataclass
class BAPQCAPState:
    """A snapshot of the vessel queue and simulated port resources."""

    pending_vessels: List[Vessel]
    berth_occupancy: List[Tuple[float, float, float, float]]
    crane_availability: List[bool]
    current_time: float
    total_vessels_served: int

    def to_vector(self, max_vessels: int = 5, max_cranes: int = 15) -> np.ndarray:
        """Encode the state as the fixed 130-feature neural-network input."""
        state_vector = np.zeros(130, dtype=np.float32)

        # Five vessel records use 50 features; the remaining blocks do not overlap them.
        for i, vessel in enumerate(self.pending_vessels[:max_vessels]):
            waiting_time = max(0.0, self.current_time - vessel.arrival_time)
            idx = i * 10
            state_vector[idx:idx + 10] = [
                vessel.id / 100,
                vessel.length / 400,
                vessel.workload / 2000,
                vessel.arrival_time / 24,
                vessel.priority / 10,
                waiting_time / 24,
                vessel.cost_per_hour / 5000,
                vessel.max_cranes / 10,
                0.0,
                0.0,
            ]

        # Each 40 m quay segment records occupancy and time until its latest release.
        for i in range(30):
            segment_start = i * 40
            segment_end = (i + 1) * 40
            occupied = 0.0
            time_until_free = 0.0
            for _, end, position, length in self.berth_occupancy:
                overlaps = not (
                    segment_end <= position or segment_start >= position + length
                )
                if overlaps and self.current_time < end:
                    occupied = 1.0
                    time_until_free = max(time_until_free, end - self.current_time)
            state_vector[50 + i * 2] = occupied
            state_vector[51 + i * 2] = time_until_free / 48

        for i, available in enumerate(self.crane_availability[:max_cranes]):
            state_vector[110 + i] = float(available)

        state_vector[125] = (self.current_time % 24) / 24
        state_vector[126] = (self.current_time % 168) / 168
        state_vector[127] = len(self.pending_vessels) / max_vessels
        state_vector[128] = self.total_vessels_served / 50
        state_vector[129] = sum(self.crane_availability) / max_cranes
        return state_vector


@dataclass
class BAPQCAPAction:
    """A discrete vessel, berth-position, and crane-count assignment."""

    vessel_id: int
    berth_position: float
    crane_count: int

    def to_index(
        self, max_vessels: int = 15, position_bins: int = 30, max_cranes: int = 5
    ) -> int:
        """Map stable vessel IDs and discretized resource choices to a Q-value index."""
        vessel_idx = self.vessel_id - 1
        position_idx = int(self.berth_position / 40)
        crane_idx = self.crane_count - 1
        if not (0 <= vessel_idx < max_vessels):
            raise ValueError("Vessel ID is outside the discrete action space.")
        if not (0 <= position_idx < position_bins):
            raise ValueError("Berth position is outside the discrete action space.")
        if not (0 <= crane_idx < max_cranes):
            raise ValueError("Crane count is outside the discrete action space.")
        return vessel_idx * position_bins * max_cranes + position_idx * max_cranes + crane_idx


@dataclass
class BAPQCAPInstance:
    """Problem parameters for the simulated port."""

    quay_length: float
    interference_factor: float
    crane_productivity: float
    num_cranes: int
    planning_horizon: float


Experience = namedtuple(
    "Experience",
    ["state", "action", "reward", "next_state", "done", "next_valid_indices"],
)

print("Data structures defined successfully.")

In [ ]:
def create_concrete_example() -> BAPQCAPInstance:
    """Create the example instance used throughout the notebook."""
    return BAPQCAPInstance(
        quay_length=1200,
        interference_factor=0.1,
        crane_productivity=30,
        num_cranes=15,
        planning_horizon=48,
    )


instance = create_concrete_example()

print("Problem instance created.")
print(f"Quay length: {instance.quay_length} m")
print(f"Crane productivity: {instance.crane_productivity} TEU/hour per crane")
print(f"Interference factor: {instance.interference_factor}")
print(f"Number of cranes: {instance.num_cranes}")
print(f"Planning horizon: {instance.planning_horizon} hours")

In [ ]:
class DuelingDQN(nn.Module):
    """Dueling Q-network with shared features and separate value and advantage streams."""

    def __init__(
        self,
        state_size: int = 130,
        action_size: int = 2250,
        hidden_sizes: Tuple[int, int, int] = (256, 128, 64),
    ):
        super().__init__()
        self.state_size = state_size
        self.action_size = action_size
        self.feature_layers = nn.Sequential(
            nn.Linear(state_size, hidden_sizes[0]),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_sizes[0], hidden_sizes[1]),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_sizes[1], hidden_sizes[2]),
            nn.ReLU(),
            nn.Dropout(0.2),
        )
        self.value_stream = nn.Sequential(
            nn.Linear(hidden_sizes[2], 64),
            nn.ReLU(),
            nn.Linear(64, 1),
        )
        self.advantage_stream = nn.Sequential(
            nn.Linear(hidden_sizes[2], 64),
            nn.ReLU(),
            nn.Linear(64, action_size),
        )

    def forward(self, state: torch.Tensor) -> torch.Tensor:
        features = self.feature_layers(state)
        value = self.value_stream(features)
        advantage = self.advantage_stream(features)
        return value + advantage - advantage.mean(dim=1, keepdim=True)


print("Dueling DQN architecture defined successfully.")

In [ ]:
class BAPQCAPEnvironment:
    """Sequential simulator for berth and crane assignment decisions."""

    def __init__(self, instance: BAPQCAPInstance):
        self.instance = instance
        self.reset()

    def reset(self) -> BAPQCAPState:
        """Start a new episode with a generated vessel set."""
        self.current_time = 0.0
        self.vessels_served = 0
        self.berth_occupancy: List[Tuple[float, float, float, float]] = []
        self.crane_availability = [True] * self.instance.num_cranes
        self.pending_vessels = self.generate_vessel_arrivals()
        return self.get_state()

    def generate_vessel_arrivals(
        self, num_vessels: int = 10, arrival_window: float = 24.0
    ) -> List[Vessel]:
        """Generate and sort a reproducible sequence of random vessel arrivals."""
        if num_vessels < 0 or num_vessels > 15:
            raise ValueError("The action encoding supports between 0 and 15 vessels.")
        vessels = []
        for i in range(num_vessels):
            vessels.append(
                Vessel(
                    id=i + 1,
                    length=np.random.uniform(200, 400),
                    workload=np.random.uniform(400, 1500),
                    arrival_time=np.random.uniform(0, arrival_window),
                    cost_per_hour=np.random.uniform(800, 1500),
                    max_cranes=np.random.randint(3, 6),
                    priority=np.random.uniform(0.5, 2.0),
                )
            )
        vessels.sort(key=lambda vessel: vessel.arrival_time)
        return vessels

    def get_state(self) -> BAPQCAPState:
        return BAPQCAPState(
            pending_vessels=self.pending_vessels.copy(),
            berth_occupancy=self.berth_occupancy.copy(),
            crane_availability=self.crane_availability.copy(),
            current_time=self.current_time,
            total_vessels_served=self.vessels_served,
        )

    def get_valid_actions(self, state: BAPQCAPState) -> List[BAPQCAPAction]:
        """Enumerate assignments that fit the quay and the discrete action space."""
        valid_actions = []
        for vessel in state.pending_vessels[:5]:
            latest_position = self.instance.quay_length - vessel.length
            for position in range(0, int(latest_position) + 1, 40):
                if not self.is_position_available(position, vessel.length, state):
                    continue
                max_cranes = min(vessel.max_cranes, sum(state.crane_availability))
                for crane_count in range(1, max_cranes + 1):
                    valid_actions.append(
                        BAPQCAPAction(vessel.id, float(position), crane_count)
                    )
        return valid_actions

    def is_position_available(
        self, position: float, length: float, state: BAPQCAPState
    ) -> bool:
        """Check for spatial overlap with vessels still occupying the berth."""
        for _, end, occupied_position, occupied_length in state.berth_occupancy:
            if state.current_time < end:
                overlaps = not (
                    position + length <= occupied_position
                    or occupied_position + occupied_length <= position
                )
                if overlaps:
                    return False
        return position >= 0 and position + length <= self.instance.quay_length

    def calculate_service_time(self, vessel: Vessel, crane_count: int) -> float:
        """Calculate service duration using diminishing productivity per crane."""
        total_productivity = sum(
            self.instance.crane_productivity
            * (1 - self.instance.interference_factor * i)
            for i in range(crane_count)
        )
        if total_productivity <= 0:
            raise ValueError("Assigned cranes must provide positive productivity.")
        return vessel.workload / total_productivity

    def step(
        self, action: BAPQCAPAction
    ) -> Tuple[BAPQCAPState, float, bool, Dict]:
        """Apply one valid assignment and return state, reward, termination, and details."""
        valid_actions = self.get_valid_actions(self.get_state())
        if action not in valid_actions:
            return self.get_state(), -1000.0, False, {"error": "Invalid action"}

        vessel = next(
            vessel for vessel in self.pending_vessels if vessel.id == action.vessel_id
        )
        start_time = max(self.current_time, vessel.arrival_time)
        service_time = self.calculate_service_time(vessel, action.crane_count)
        finish_time = start_time + service_time
        waiting_time = start_time - vessel.arrival_time
        total_time = waiting_time + service_time
        utilization_bonus = 500.0 if action.crane_count >= 3 else 0.0
        if waiting_time < 2:
            utilization_bonus += 300.0
        reward = -vessel.cost_per_hour * total_time + utilization_bonus

        self.berth_occupancy.append(
            (start_time, finish_time, action.berth_position, vessel.length)
        )
        self.pending_vessels.remove(vessel)
        self.vessels_served += 1
        # The sequential model releases all assigned cranes before the next decision.
        self.current_time = finish_time
        done = not self.pending_vessels or self.current_time > self.instance.planning_horizon
        info = {
            "vessel_id": vessel.id,
            "waiting_time": waiting_time,
            "service_time": service_time,
            "total_time": total_time,
            "utilization_bonus": utilization_bonus,
            "finish_time": finish_time,
        }
        return self.get_state(), reward, done, info


print("Environment class defined successfully.")

In [ ]:
class BAPQCAPAgent:
    """Dueling DQN agent with valid-action masking and experience replay."""

    def __init__(
        self,
        state_size: int = 130,
        action_size: int = 2250,
        learning_rate: float = 0.001,
    ):
        self.state_size = state_size
        self.action_size = action_size
        self.learning_rate = learning_rate
        self.memory = deque(maxlen=100000)
        self.epsilon = 1.0
        self.epsilon_min = 0.01
        self.epsilon_decay = 0.995
        self.gamma = 0.95
        self.q_network = DuelingDQN(state_size, action_size)
        self.target_network = DuelingDQN(state_size, action_size)
        self.optimizer = optim.Adam(self.q_network.parameters(), lr=learning_rate)
        self.update_target_network()
        self.training_loss: List[float] = []
        self.q_value_history: List[float] = []

    def update_target_network(self) -> None:
        """Synchronize the target network used to form Bellman targets."""
        self.target_network.load_state_dict(self.q_network.state_dict())
        self.target_network.eval()

    def get_valid_action_indices(
        self, state: BAPQCAPState, env: BAPQCAPEnvironment
    ) -> List[int]:
        return [
            action.to_index()
            for action in env.get_valid_actions(state)
        ]

    def act(
        self, state: BAPQCAPState, env: BAPQCAPEnvironment, training: bool = True
    ) -> Optional[BAPQCAPAction]:
        valid_actions = env.get_valid_actions(state)
        if not valid_actions:
            return None
        if training and np.random.random() <= self.epsilon:
            return random.choice(valid_actions)

        state_vector = torch.as_tensor(state.to_vector(), dtype=torch.float32).unsqueeze(0)
        was_training = self.q_network.training
        self.q_network.eval()
        with torch.no_grad():
            q_values = self.q_network(state_vector)[0]
        if was_training:
            self.q_network.train()

        valid_indices = self.get_valid_action_indices(state, env)
        best_index = max(valid_indices, key=lambda index: q_values[index].item())
        return next(action for action in valid_actions if action.to_index() == best_index)

    def remember(
        self,
        state: BAPQCAPState,
        action: BAPQCAPAction,
        reward: float,
        next_state: BAPQCAPState,
        done: bool,
        next_valid_indices: List[int],
    ) -> None:
        self.memory.append(
            Experience(state, action, reward, next_state, done, next_valid_indices)
        )

    def replay(self, batch_size: int = 32) -> None:
        """Update Q-values from a random batch of stored transitions."""
        if len(self.memory) < batch_size:
            return
        batch = random.sample(self.memory, batch_size)
        states = torch.as_tensor(
            np.stack([item.state.to_vector() for item in batch]), dtype=torch.float32
        )
        actions = torch.as_tensor(
            [item.action.to_index() for item in batch], dtype=torch.long
        )
        rewards = torch.as_tensor([item.reward for item in batch], dtype=torch.float32)
        next_states = torch.as_tensor(
            np.stack([item.next_state.to_vector() for item in batch]),
            dtype=torch.float32,
        )
        dones = torch.as_tensor([item.done for item in batch], dtype=torch.float32)

        self.q_network.train()
        current_q = self.q_network(states).gather(1, actions.unsqueeze(1)).squeeze(1)
        targets = rewards.clone()
        nonterminal_rows = [
            i for i, item in enumerate(batch) if not item.done and item.next_valid_indices
        ]
        if nonterminal_rows:
            self.target_network.eval()
            with torch.no_grad():
                next_q = self.target_network(next_states[nonterminal_rows])
                for row, batch_index in enumerate(nonterminal_rows):
                    valid_indices = batch[batch_index].next_valid_indices
                    targets[batch_index] += (
                        self.gamma * next_q[row, valid_indices].max()
                    )

        loss = F.mse_loss(current_q, targets)
        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.q_network.parameters(), 1.0)
        self.optimizer.step()
        self.training_loss.append(float(loss.item()))
        self.q_value_history.append(float(current_q.detach().mean().item()))

    def decay_epsilon(self) -> None:
        """Reduce exploration gradually while keeping a nonzero exploration floor."""
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)


print("DQN agent defined successfully.")

In [ ]:
def train_dqn_agent(
    instance: BAPQCAPInstance,
    num_episodes: int = 1000,
    max_steps_per_episode: int = 100,
) -> Tuple[BAPQCAPAgent, BAPQCAPEnvironment, Dict]:
    """Train the agent and return it with its environment and recorded metrics."""
    if num_episodes <= 0 or max_steps_per_episode <= 0:
        raise ValueError("Episode and step limits must be positive.")

    print("Training the Dueling DQN agent.")
    print(f"Episodes: {num_episodes}")
    print(f"Maximum steps per episode: {max_steps_per_episode}")
    env = BAPQCAPEnvironment(instance)
    agent = BAPQCAPAgent()
    episode_rewards = []
    episode_lengths = []
    epsilon_history = []
    vessels_served_history = []
    start_time = time.time()

    for episode in range(num_episodes):
        state = env.reset()
        total_reward = 0.0
        steps = 0
        vessels_served = 0

        for _ in range(max_steps_per_episode):
            action = agent.act(state, env, training=True)
            if action is None:
                break
            next_state, reward, done, info = env.step(action)
            next_valid_indices = agent.get_valid_action_indices(next_state, env)
            agent.remember(
                state,
                action,
                reward,
                next_state,
                done,
                next_valid_indices,
            )
            state = next_state
            total_reward += reward
            steps += 1
            vessels_served += int("vessel_id" in info)
            if done:
                break

        if len(agent.memory) > 1000:
            agent.replay(batch_size=32)
        agent.decay_epsilon()
        if (episode + 1) % 100 == 0:
            agent.update_target_network()

        episode_rewards.append(total_reward)
        episode_lengths.append(steps)
        epsilon_history.append(agent.epsilon)
        vessels_served_history.append(vessels_served)
        if episode % 100 == 0 or episode == num_episodes - 1:
            recent_rewards = episode_rewards[-100:]
            print(
                f"Episode {episode + 1:4d}: reward={total_reward:9.1f}, "
                f"recent mean={np.mean(recent_rewards):9.1f}, "
                f"steps={steps:3d}, epsilon={agent.epsilon:.3f}, "
                f"vessels served={vessels_served}"
            )

    training_time = time.time() - start_time
    training_results = {
        "episode_rewards": episode_rewards,
        "episode_lengths": episode_lengths,
        "epsilon_history": epsilon_history,
        "vessels_served_history": vessels_served_history,
        "training_time": training_time,
        "final_epsilon": agent.epsilon,
        "training_loss": agent.training_loss.copy(),
        "q_value_history": agent.q_value_history.copy(),
    }
    print("Training complete.")
    print(f"Training time: {training_time:.2f} seconds")
    print(f"Final exploration rate: {agent.epsilon:.3f}")
    print(f"Mean reward over the final episodes: {np.mean(episode_rewards[-100:]):.1f}")
    print(
        "Mean vessels served over the final episodes: "
        f"{np.mean(vessels_served_history[-100:]):.1f}"
    )
    return agent, env, training_results


agent, env, training_results = train_dqn_agent(
    instance, num_episodes=500, max_steps_per_episode=50
)

In [ ]:
def visualize_drl_results(training_results: Dict) -> None:
    """Plot training diagnostics and report observed episode statistics."""
    rewards = training_results["episode_rewards"]
    episodes = np.arange(1, len(rewards) + 1)
    window_size = 50
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle("Dueling DQN Training Diagnostics", fontsize=16, fontweight="bold")

    ax = axes[0, 0]
    ax.plot(episodes, rewards, color="tab:blue", alpha=0.35, label="Episode reward")
    if len(rewards) >= window_size:
        moving_average = np.convolve(
            rewards, np.ones(window_size) / window_size, mode="valid"
        )
        ax.plot(
            np.arange(window_size, len(rewards) + 1),
            moving_average,
            color="tab:red",
            label=f"Moving average ({window_size} episodes)",
        )
    ax.set(title="Episode rewards", xlabel="Episode", ylabel="Reward (cost units)")
    ax.grid(True, alpha=0.3)
    ax.legend()

    ax = axes[0, 1]
    ax.plot(episodes, training_results["epsilon_history"], color="tab:green")
    ax.set(title="Exploration rate", xlabel="Episode", ylabel="Epsilon")
    ax.set_ylim(0, 1.05)
    ax.grid(True, alpha=0.3)

    ax = axes[1, 0]
    losses = training_results["training_loss"]
    if losses:
        ax.plot(np.arange(1, len(losses) + 1), losses, color="tab:purple", alpha=0.7)
    ax.set(title="Replay training loss", xlabel="Update", ylabel="Mean squared TD error")
    ax.grid(True, alpha=0.3)

    ax = axes[1, 1]
    served = training_results["vessels_served_history"]
    ax.plot(episodes, served, color="tab:orange", alpha=0.7, label="Episode count")
    if len(served) >= window_size:
        served_average = np.convolve(
            served, np.ones(window_size) / window_size, mode="valid"
        )
        ax.plot(
            np.arange(window_size, len(served) + 1),
            served_average,
            color="tab:brown",
            label=f"Moving average ({window_size} episodes)",
        )
    ax.set(title="Vessels served", xlabel="Episode", ylabel="Vessels")
    ax.grid(True, alpha=0.3)
    ax.legend()

    plt.tight_layout()
    plt.show()
    print(f"Episodes recorded: {len(rewards)}")
    print(f"Training time: {training_results['training_time']:.2f} seconds")
    print(f"Mean reward in the final 100 episodes: {np.mean(rewards[-100:]):.1f}")
    print(f"Best observed episode reward: {max(rewards):.1f}")
    print(f"Final exploration rate: {training_results['final_epsilon']:.3f}")


visualize_drl_results(training_results)

In [ ]:
def evaluate_trained_agent(
    agent: BAPQCAPAgent,
    env: BAPQCAPEnvironment,
    num_episodes: int = 100,
) -> Tuple[Dict, Dict]:
    """Measure the greedy policy on newly generated episodes."""
    if num_episodes <= 0:
        raise ValueError("The number of evaluation episodes must be positive.")
    metrics = {
        "episode_rewards": [],
        "vessels_served": [],
        "waiting_times": [],
        "service_times": [],
        "total_times": [],
        "crane_counts": [],
        "crane_hours": [],
        "berth_length_hours": [],
    }

    for _ in range(num_episodes):
        state = env.reset()
        episode_reward = 0.0
        served = 0
        for _ in range(100):
            action = agent.act(state, env, training=False)
            if action is None:
                break
            next_state, reward, done, info = env.step(action)
            episode_reward += reward
            served += int("vessel_id" in info)
            if "vessel_id" in info:
                vessel = next(
                    item for item in env.berth_occupancy
                    if item[2] == action.berth_position and item[1] == info["finish_time"]
                )
                length_hours = vessel[3] * info["service_time"]
                metrics["waiting_times"].append(info["waiting_time"])
                metrics["service_times"].append(info["service_time"])
                metrics["total_times"].append(info["total_time"])
                metrics["crane_counts"].append(action.crane_count)
                metrics["crane_hours"].append(action.crane_count * info["service_time"])
                metrics["berth_length_hours"].append(length_hours)
            state = next_state
            if done:
                break
        metrics["episode_rewards"].append(episode_reward)
        metrics["vessels_served"].append(served)

    def safe_mean(values: List[float]) -> float:
        return float(np.mean(values)) if values else float("nan")

    summary = {
        "avg_reward": safe_mean(metrics["episode_rewards"]),
        "std_reward": float(np.std(metrics["episode_rewards"])),
        "avg_vessels_served": safe_mean(metrics["vessels_served"]),
        "avg_waiting_time": safe_mean(metrics["waiting_times"]),
        "avg_service_time": safe_mean(metrics["service_times"]),
        "avg_total_time": safe_mean(metrics["total_times"]),
        "avg_cranes_used": safe_mean(metrics["crane_counts"]),
        "crane_utilization_percent": (
            100 * sum(metrics["crane_hours"])
            / (env.instance.num_cranes * env.instance.planning_horizon)
        ),
        "berth_space_time_utilization_percent": (
            100 * sum(metrics["berth_length_hours"])
            / (env.instance.quay_length * env.instance.planning_horizon)
        ),
        "total_vessels_processed": int(sum(metrics["vessels_served"])),
    }
    print(f"Evaluation episodes: {num_episodes}")
    print(f"Mean reward: {summary['avg_reward']:.1f} ± {summary['std_reward']:.1f}")
    print(f"Mean vessels served per episode: {summary['avg_vessels_served']:.1f}")
    print(f"Mean waiting time: {summary['avg_waiting_time']:.2f} hours")
    print(f"Mean service time: {summary['avg_service_time']:.2f} hours")
    print(f"Mean turnaround time: {summary['avg_total_time']:.2f} hours")
    print(f"Mean cranes assigned per vessel: {summary['avg_cranes_used']:.2f}")
    print(f"Crane utilization: {summary['crane_utilization_percent']:.2f}%")
    print(
        "Berth space-time utilization: "
        f"{summary['berth_space_time_utilization_percent']:.2f}%"
    )
    print(f"Vessels processed: {summary['total_vessels_processed']}")
    return metrics, summary


eval_metrics, eval_summary = evaluate_trained_agent(agent, env, num_episodes=50)

In [ ]:
def analyze_learned_policy(
    agent: BAPQCAPAgent, env: BAPQCAPEnvironment
) -> List[Dict]:
    """Inspect actions selected in scenarios with different arrival windows."""
    test_scenarios = [
        {"vessels": 5, "arrival_window": 48.0, "description": "Spread arrivals"},
        {"vessels": 10, "arrival_window": 24.0, "description": "Standard arrivals"},
        {"vessels": 15, "arrival_window": 12.0, "description": "Concentrated arrivals"},
    ]
    scenario_results = []
    for scenario in test_scenarios:
        test_env = BAPQCAPEnvironment(env.instance)
        test_env.pending_vessels = test_env.generate_vessel_arrivals(
            scenario["vessels"], scenario["arrival_window"]
        )
        state = test_env.get_state()
        actions_taken = []
        rewards = []
        for _ in range(20):
            action = agent.act(state, test_env, training=False)
            if action is None:
                break
            state, reward, done, _ = test_env.step(action)
            actions_taken.append(action)
            rewards.append(reward)
            if done:
                break
        result = {
            "scenario": scenario["description"],
            "vessels": scenario["vessels"],
            "actions": actions_taken,
            "actions_taken": len(actions_taken),
            "average_cranes": float(np.mean([a.crane_count for a in actions_taken]))
            if actions_taken else float("nan"),
            "average_position": float(np.mean([a.berth_position for a in actions_taken]))
            if actions_taken else float("nan"),
            "total_reward": float(sum(rewards)),
            "average_reward": float(np.mean(rewards)) if rewards else float("nan"),
        }
        scenario_results.append(result)
        print(
            f"{result['scenario']}: {result['actions_taken']} actions, "
            f"mean reward {result['average_reward']:.1f}"
        )

    all_actions = [
        action for result in scenario_results for action in result["actions"]
    ]
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle("Policy behavior in generated scenarios", fontsize=16, fontweight="bold")

    ax = axes[0, 0]
    if all_actions:
        ax.hist(
            [action.crane_count for action in all_actions],
            bins=np.arange(0.5, 6.5, 1),
            edgecolor="black",
        )
    ax.set(title="Crane-count choices", xlabel="Crane count", ylabel="Frequency")
    ax.grid(True, alpha=0.3)

    ax = axes[0, 1]
    if all_actions:
        ax.hist(
            [action.berth_position for action in all_actions],
            bins=20,
            edgecolor="black",
        )
    ax.set(title="Berth-position choices", xlabel="Position (m)", ylabel="Frequency")
    ax.grid(True, alpha=0.3)

    ax = axes[1, 0]
    labels = [result["scenario"] for result in scenario_results]
    ax.bar(labels, [result["average_reward"] for result in scenario_results])
    ax.set(title="Mean reward by scenario", xlabel="Scenario", ylabel="Mean reward")
    ax.tick_params(axis="x", rotation=15)
    ax.grid(True, axis="y", alpha=0.3)

    ax = axes[1, 1]
    ax.bar(labels, [result["actions_taken"] for result in scenario_results])
    ax.set(title="Actions taken by scenario", xlabel="Scenario", ylabel="Actions")
    ax.tick_params(axis="x", rotation=15)
    ax.grid(True, axis="y", alpha=0.3)

    plt.tight_layout()
    plt.show()
    return scenario_results


policy_analysis = analyze_learned_policy(agent, env)

## Why use a Dueling DQN here?

The Dueling DQN structure represents a state value and action-specific advantages in separate network streams. This can be useful when many actions have similar effects in a state, because the value stream can learn what makes the state favorable while the advantage stream distinguishes assignments. Experience replay, a target network, and epsilon-greedy exploration are included as parts of the DQN training procedure.

### Strengths and limitations

**Strengths**

- The agent can represent a policy over a fixed, discretized state and action space.
- Experience replay reuses sampled transitions, while periodic target-network synchronization provides a more stable reference for Bellman targets.
- Valid-action masking prevents the policy and its bootstrap target from selecting assignments that the simulator disallows.

**Limitations**

- Neural-network training can be sensitive to initialization, reward scales, and hyperparameters; a plotted learning curve is not proof of convergence.
- The fixed-size state representation omits detailed information about pending vessels beyond the first five.
- The sequential simulator does not represent concurrent berth service or crane scheduling over time, so its results should not be interpreted as operational performance for a full terminal.
- The learned policy is not guaranteed to be globally optimal, and this notebook does not establish superiority over a baseline.

### When this approach is worth studying

A DQN is a useful learning framework when decisions are sequential and a state-action value function is of interest. The present example is a compact demonstration of that framework. Applying it to operational planning would require a state representation and transition model that faithfully capture the terminal's resource interactions, along with evaluation on appropriate scenarios and baselines.

In [ ]:
def final_drl_summary(
    agent: BAPQCAPAgent,
    training_results: Dict,
    eval_summary: Dict,
    policy_analysis: List[Dict],
) -> None:
    """Summarize the implemented method and measurements from this run."""
    rewards = training_results["episode_rewards"]
    print("Dueling DQN implementation summary")
    print(f"State features: {agent.state_size}")
    print(f"Discrete action indices: {agent.action_size}")
    print(f"Stored transitions: {len(agent.memory)}")
    print(f"Final exploration rate: {agent.epsilon:.3f}")
    print(f"Training episodes: {len(rewards)}")
    print(f"Training time: {training_results['training_time']:.2f} seconds")
    print(f"Best observed episode reward: {max(rewards):.1f}")
    print(f"Evaluation mean reward: {eval_summary['avg_reward']:.1f}")
    print(
        "Evaluation mean vessels served per episode: "
        f"{eval_summary['avg_vessels_served']:.1f}"
    )
    print(f"Policy-analysis scenarios: {len(policy_analysis)}")
    print(
        "Interpretation: these values describe the generated episodes and the "
        "sequential simulator; they do not establish convergence or optimality."
    )


final_drl_summary(agent, training_results, eval_summary, policy_analysis)